In [ ]:
import sys; sys.path.append("..")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit

from src.config import TRAIN_SEASONS
from src.features.build import add_elo
from src.evaluation.metrics import log_loss_hda, to_hda

m = pd.read_csv("../data/processed/matches.csv", parse_dates=["date"])
m["season"] = m["season"].astype(str).str.zfill(4)
print(len(m))


In [ ]:
def add_h2h(df, window=5):
    """h2h_score: media de (+1 gana, 0 empata, -1 pierde) del equipo LOCAL de hoy
    en sus últimos `window` enfrentamientos previos contra este rival. 0 si no hay historial.
    h2h_n: cuántos enfrentamientos previos hay (0..window)."""
    hist = {}                      # (equipo, equipo) → lista de (local de aquel partido, resultado desde el local)
    filas = []
    for r in df.sort_values(["date", "match_id"]).itertuples():
        key = frozenset((r.home, r.away))
        past = hist.get(key, [])[-window:]                       # SOLO partidos anteriores
        s = [sc if h == r.home else -sc for h, sc in past]       # desde la perspectiva del local de hoy
        filas.append((r.match_id, float(np.mean(s)) if s else 0.0, len(s)))
        hist.setdefault(key, []).append((r.home, {"H": 1, "D": 0, "A": -1}[r.result]))
    f = pd.DataFrame(filas, columns=["match_id", "h2h_score", "h2h_n"])
    return df.merge(f, on="match_id", how="left")

h = add_h2h(add_elo(m))
print(h[["h2h_score", "h2h_n"]].describe().round(2))
print(h["h2h_n"].value_counts().sort_index())

In [ ]:
h["h2h_w"] = h["h2h_score"] * h["h2h_n"] / 5      # versión con peso: menos historial → menos influencia

def eval_cols(e, cols, n_splits=5):
    tr = (e[e["season"].isin(TRAIN_SEASONS)]
            .sort_values(["date", "match_id"]).reset_index(drop=True))
    X = tr[cols].to_numpy().astype(float)
    X[:, 0] = X[:, 0] / 100                       # la primera columna siempre es elo_diff
    y = tr["result"].to_numpy()
    p = np.full((len(tr), 3), np.nan)
    por_bloque = []
    for a, b in TimeSeriesSplit(n_splits=n_splits).split(X):
        clf = LogisticRegression().fit(X[a], y[a])
        p[b] = to_hda(clf, clf.predict_proba(X[b]))
        por_bloque.append(log_loss_hda(y[b], p[b]))
    ok = ~np.isnan(p[:, 0])
    return log_loss_hda(y[ok], p[ok]), np.array(por_bloque)

base, b0 = eval_cols(h, ["elo_diff"])
a1, b1 = eval_cols(h, ["elo_diff", "h2h_score"])
a2, b2 = eval_cols(h, ["elo_diff", "h2h_w"])
a3, b3 = eval_cols(h, ["elo_diff", "h2h_score", "h2h_n"])

print(f"Solo Elo:               {base:.4f}")
print(f"Elo + h2h_score:        {a1:.4f}   (cambio {a1-base:+.4f})")
print(f"Elo + h2h_w (con peso): {a2:.4f}   (cambio {a2-base:+.4f})")
print(f"Elo + h2h_score + n:    {a3:.4f}   (cambio {a3-base:+.4f})")
print("\nDiferencia por bloque temporal (negativo = H2H ayuda):")
print("score:", np.round(b1 - b0, 4))
print("peso: ", np.round(b2 - b0, 4))